# Verify Confidence-Guided Token Rejection

**Before running:** Runtime > Change runtime type > **A100 GPU**.

This notebook runs the three verification tests for the rejection mechanism:
1. **Parity test** (hard gate) — `generate_with_rejection(threshold=0)` must match vanilla `generate()` bit-for-bit across all 3 confidence metrics.
2. **Empty-ids forward test** — validates the tricky forward path used by the refinement ablation.
3. **End-to-end smoke test** — 16 samples through the full pipeline with debug asserts enabled.

All three must pass before we run the Phase 1 pilot sweep (18 configurations).

Total runtime: ~5 minutes.

## 1. Verify GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > A100 GPU")

## 2. Mount Drive & get the latest code

Re-uses the same `ARPG-assets/` folder as the baseline notebook (weights + eval data already downloaded).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ARPG = "/content/drive/MyDrive/ARPG-assets"
!mkdir -p {DRIVE_ARPG}/weights {DRIVE_ARPG}/eval {DRIVE_ARPG}/external {DRIVE_ARPG}/results

In [ ]:
# Clone (or re-clone) from GitHub
!rm -rf /content/ARPG
!git clone https://github.com/rshahbazov23/comp447-arpg-private.git /content/ARPG
%cd /content/ARPG

# Symlink heavy assets from Drive
!ln -sfn {DRIVE_ARPG}/weights weights
!ln -sfn {DRIVE_ARPG}/eval eval
!ln -sfn {DRIVE_ARPG}/external external
!ln -sfn {DRIVE_ARPG}/results results

# Install runtime deps
!pip install -q transformers einops Pillow tqdm numpy

In [ ]:
# Sanity-check: assets + new scripts should be present
import os
expected = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "models/arpg.py",
    "models/confidence.py",
    "utils/rejection_tracker.py",
    "scripts/test_rejection_parity.py",
    "scripts/test_empty_ids_forward.py",
    "scripts/run_rejection_smoke.sh",
]
for path in expected:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {path}")

# If anything is MISSING, the most likely cause is that `git pull` didn't
# fetch the merged PR. Check: !git log --oneline -5

## 3. Test 1: Parity (hard gate)

This is the most important test. Runs ~60 seconds.

`generate_with_rejection(threshold=0.0, max_reject_rate=0.0)` must produce **bit-identical** output to `generate()` under the same RNG seed, for all 3 confidence metrics.

If this fails, the KV-cache or position bookkeeping is broken. Stop and debug before running anything else.

In [ ]:
!python scripts/test_rejection_parity.py --gpt-ckpt weights/arpg_300m.pt

**Expected output:** `All parity checks passed.`

If any check fails, the error message points to the first differing positions or the specific checkpoint that broke.

## 4. Test 2: Empty-ids forward

Validates the cross-attention-only path used by `generate_with_refinement`. Runs ~30 seconds.

Three sub-checks:
1. Empty-ids forward produces correct-shape finite logits on a populated cache.
2. End-to-end `generate_with_refinement(k=0.1)` runs without crashing.
3. `refinement_k=0.0` is a bit-identical no-op vs vanilla.

In [ ]:
!python scripts/test_empty_ids_forward.py --gpt-ckpt weights/arpg_300m.pt

**Expected output:** `All empty-ids path tests passed.`

## 5. Test 3: End-to-end smoke test

Generates 16 samples with rejection enabled (τ=0.5, max_reject=0.2, max_prob metric) and all debug asserts on. Runs ~60-90 seconds.

Produces:
- 16 generated PNGs
- `log.json` — per-step tracker data
- `log_heatmap.png` — 16×16 spatial rejection heatmap

In [ ]:
# Clean any previous smoke test output
!rm -rf /content/ARPG/samples/rejection-smoke

# Run the smoke test (local samples dir for fast I/O)
!SAMPLE_DIR=/content/ARPG/samples/rejection-smoke bash scripts/run_rejection_smoke.sh

### Inspect: spatial rejection heatmap

Shows where on the 16×16 token grid the rejections concentrated. A non-uniform pattern is expected (some regions are harder to predict than others).

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

heatmap = Image.open('/content/ARPG/samples/rejection-smoke/log_heatmap.png')
plt.figure(figsize=(6, 6))
plt.imshow(heatmap)
plt.axis('off')
plt.title('Rejection heatmap (16 samples, tau=0.5, max_prob)')
plt.show()

### Inspect: generated samples

The PNGs should look like reasonable ImageNet class-conditional images (not noise).

In [ ]:
import glob
sample_dir = glob.glob('/content/ARPG/samples/rejection-smoke/ARPG-L-*')[0]
print(f"Sample folder: {sample_dir}\n")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = Image.open(f'{sample_dir}/{i:06d}.png')
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Sample {i}')
plt.suptitle('First 8 generated samples (rejection mode)', fontsize=14)
plt.tight_layout()
plt.show()

### Inspect: tracker log summary

In [ ]:
import json

with open('/content/ARPG/samples/rejection-smoke/log.json') as f:
    log = json.load(f)

print("=== Summary ===")
for k, v in log['summary'].items():
    if k == 'mean_commit_step_per_position':
        continue  # 256-element list, skip
    print(f"  {k}: {v}")

print(f"\nFirst-batch step detail: {len(log['steps_first_batch'])} entries")
first_step = log['steps_first_batch'][0]
print(f"  step 0: {first_step['num_pred']} attempted, "
      f"{first_step['num_accepted']} accepted, "
      f"{first_step['num_rejected']} rejected")
last_step = log['steps_first_batch'][-1]
print(f"  step {last_step['step']}: {last_step['num_pred']} attempted, "
      f"{last_step['num_accepted']} accepted, "
      f"{last_step['num_rejected']} rejected (final step, should accept all)")

### Per-step rejection rate curve

The proposal mentions per-step rejection rate curves as part of the analysis (§4). This plot is for the first batch only but gives a qualitative picture.

In [ ]:
steps = log['steps_first_batch']
rejection_rates = [s['num_rejected'] / max(1, s['num_pred']) for s in steps]
num_attempted = [s['num_pred'] for s in steps]

fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = 'tab:blue'
ax1.set_xlabel('decoding step')
ax1.set_ylabel('rejection rate', color=color1)
ax1.plot(range(len(rejection_rates)), rejection_rates, color=color1, marker='o', markersize=3)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(0, max(rejection_rates) * 1.1 if max(rejection_rates) > 0 else 0.3)

ax2 = ax1.twinx()
color2 = 'tab:orange'
ax2.set_ylabel('num positions attempted', color=color2)
ax2.plot(range(len(num_attempted)), num_attempted, color=color2, marker='s', markersize=3)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Per-step rejection rate and positions attempted (first batch)')
fig.tight_layout()
plt.show()

## 6. Verdict

If all three tests passed and the generated samples look reasonable, the implementation is verified and ready for the Phase 1 pilot sweep.

**Next step:** run the 18-config pilot grid per the proposal (τ × max_reject × metric), evaluate with FID-10K, and identify 3-5 promising configurations for the full FID-50K runs in Phase 2.

**If a test failed:** the error output points to the failure mode. Share it with the team and we'll debug.

## 7. (Optional) Backup results to Drive

In [ ]:
# Uncomment to save the smoke test output to Drive for record-keeping
# !mkdir -p {DRIVE_ARPG}/results/rejection-smoke-$(date +%Y%m%d)
# !cp -r /content/ARPG/samples/rejection-smoke/* {DRIVE_ARPG}/results/rejection-smoke-$(date +%Y%m%d)/
# print('Saved to Drive')